In [ ]:
#OTTIENI YOLO11N.PT DA ULTRALYTICS
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n.pt") #esporta in .pt

In [31]:
model= YOLO("yolo11micro.pt")
model.info()


YOLO11micro summary: 231 layers, 864,600 parameters, 864,584 gradients, 2.7 GFLOPs


(231, 864600, 864584, 2.6902144)

In [ ]:
#Importa modello YOLO personalizzato/custom da file Yaml
from ultralytics import YOLO

model = YOLO("yolo11micro.yaml")
model.info()
model.save("yolo11micro.pt")



YOLO11micro summary: 231 layers, 864,600 parameters, 864,584 gradients, 2.7 GFLOPs


In [ ]:
#MOSTRA TUTTI I LAYER CONV2D E BATCHNORM2D DEL MODELLO
import torch
import torch.nn as nn
from ultralytics import YOLO

# Carica il modello
#model = YOLO("yolo11n.pt")

# Identifica i primi layer
print("=== PRIMI LAYER DEL MODELLO ===")
for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.BatchNorm2d)):
        if 'model.0' in name or 'model.1' in name:
            if isinstance(module, nn.Conv2d):
                print(f"{name}: Conv2d(in_channels={module.in_channels}, out_channels={module.out_channels})")
            elif isinstance(module, nn.BatchNorm2d):
                print(f"{name}: BatchNorm2d(num_features={module.num_features})")

In [236]:
# stampa summary completo con layers, parametri, GFLOPs ecc.
model = YOLO("yolo11s")
model.info()

YOLO11s summary: 181 layers, 9,458,752 parameters, 0 gradients, 21.7 GFLOPs


(181, 9458752, 0, 21.718374400000002)

In [ ]:
#Guarda i modules pytorch (model.0 ... model.23) di yolo11n.pt
#model = YOLO("yolo11n.pt")
model = YOLO("yolo11n.pt")
#stampa numero di parametri totali
# Conta i parametri
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f"Parametri totali: {total_params:,}")
print(f"Parametri allenabili: {trainable_params:,}")
print(f"Parametri non allenabili: {total_params - trainable_params:,}")
for m in model.modules():
    print(m)

In [274]:
#import torch
from ultralytics import YOLO
model = YOLO("yolo11micro.pt")

results = model.predict(
    "../../Calibrazione/soloESP/sfoca.jpg",
    save=True,
    project="../../FotoInference",
    imgsz=640,
    conf=0.4,
    name="detection"
)



image 1/1 /Users/alessioprato/Desktop/Tesi Nuova/ESP32CAM_ESPIDF/Notebooks/../../Calibrazione/soloESP/sfoca.jpg: 480x640 (no detections), 70.7ms
Speed: 7.5ms preprocess, 70.7ms inference, 3.4ms postprocess per image at shape (1, 3, 480, 640)
Results saved to ../../FotoInference/detection46


In [95]:
#inferenza per modello standard
model = YOLO("epoch130.pt")
nomeImmagine = "../../Foto/testIO.jpg"

results = model.predict(
    nomeImmagine, 
    imgsz=64,
    save=True,  
    project="../../FotoInference",
    name="detection",
)



image 1/1 /Users/alessioprato/Desktop/Tesi Nuova/ESP32CAM_ESPIDF/Notebooks/../../Foto/testIO.jpg: 64x64 1 person, 5.8ms
Speed: 0.4ms preprocess, 5.8ms inference, 0.8ms postprocess per image at shape (1, 3, 64, 64)
Results saved to ../../FotoInference/detection17


In [ ]:
# STAMPA TUTTI I LAYER E BLOCCHI DEL MODELLO
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

trainable = 0
total = 0
for i, (name, p) in enumerate(model.model.named_parameters()):
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()
    if i < 1000:
        print(i, name, p.requires_grad, p.numel())
print(f"Trainable params: {trainable:,} / Total params: {total:,} ({100*trainable/total:.2f}%)")


In [108]:
#ESPORTA YOLO .PT IN .ONNX TRAMITE SCRIPT DI EXPRESSIF (APPORTANDO MODIFICHE AD ALCUNI LAYERS)
from ultralytics import YOLO
from ultralytics.nn.modules import Detect, Attention
from ultralytics.engine.exporter import Exporter, try_export, arange_patch
from ultralytics.utils import LOGGER, __version__, colorstr
from ultralytics.utils.checks import check_requirements
from ultralytics.utils.torch_utils import get_latest_opset
import torch
import onnx


class ESP_Detect(Detect):
    def forward(self, x):
        """Returns predicted bounding boxes and class probabilities respectively."""
        # self.nl = 3
        box0 = self.cv2[0](x[0])
        score0 = self.cv3[0](x[0])

        box1 = self.cv2[1](x[1])
        score1 = self.cv3[1](x[1])

        box2 = self.cv2[2](x[2])
        score2 = self.cv3[2](x[2])

        return box0, score0, box1, score1, box2, score2


class ESP_Attention(Attention):
    def forward(self, x):
        """
        Forward pass of the Attention module.

        Args:
            x (torch.Tensor): The input tensor.

        Returns:
            (torch.Tensor): The output tensor after self-attention.
        """
        B, C, H, W = x.shape
        N = H * W
        qkv = self.qkv(x)
        q, k, v = qkv.view(
            -1, self.num_heads, self.key_dim * 2 + self.head_dim, N
        ).split([self.key_dim, self.key_dim, self.head_dim], dim=2)
        attn = (q.transpose(-2, -1) @ k) * self.scale
        attn = attn.softmax(dim=-1)
        x = (v @ attn.transpose(-2, -1)).view(-1, C, H, W) + self.pe(
            v.reshape(-1, C, H, W)
        )
        x = self.proj(x)
        return x


class ESP_Detect_Exporter(Exporter):
    """
    adapted from ultralytics for detection task
    """

    @try_export
    def export_onnx(self, prefix=colorstr("ONNX:")):
        """YOLO ONNX export."""
        requirements = ["onnx>=1.14.0"]  # from esp-ppq requirments.txt
        # since onnxslim will cause NCHW -> 1(N*C)HW in yolo11, we replace onnxslim with onnxsim
        if self.args.simplify:
            requirements += [
                "onnxsim",
                "onnxruntime" + ("-gpu" if torch.cuda.is_available() else ""),
            ]
        check_requirements(requirements)

        opset_version = self.args.opset or get_latest_opset()
        LOGGER.info(
            f"\n{prefix} starting export with onnx {onnx.__version__} opset {opset_version}..."
        )
        f = str(self.file.with_suffix(".onnx"))
        output_names = ["box0", "score0", "box1", "score1", "box2", "score2"]
        dynamic = (
            self.args.dynamic
        )  # case 1: deploy model on ESP32, dynamic=False; case 2: QAT gt onnx for inference, dynamic=True
        if dynamic:
            dynamic = {"images": {0: "batch"}}
            for name in output_names:
                dynamic[name] = {0: "batch"}

        with arange_patch(self.args):
            torch.onnx.export(
                self.model,
                self.im,
                f,
                verbose=False,
                opset_version=opset_version,
                do_constant_folding=False, #Default era false
                input_names=["images"],
                output_names=output_names,
                dynamic_axes=dynamic or None,
            )
        # Checks
        model_onnx = onnx.load(f)  # load onnx model

        # Simplify
        if self.args.simplify:
            try:
                import onnxsim

                LOGGER.info(
                    f"{prefix} simplifying with onnxsim {onnxsim.__version__}..."
                )
                model_onnx, _ = onnxsim.simplify(model_onnx)

            except Exception as e:
                LOGGER.warning(f"{prefix} simplifier failure: {e}")

        # Metadata
        for k, v in self.metadata.items():
            meta = model_onnx.metadata_props.add()
            meta.key, meta.value = k, str(v)

        onnx.save(model_onnx, f)
        return f, model_onnx


class ESP_YOLO(YOLO):
    def export(
        self,
        **kwargs,
    ):
        self._check_is_pytorch_model()
        custom = {
            "imgsz": self.model.args["imgsz"],
            "batch": 1,
            "data": None,
            "device": None,
            "verbose": False,
        }
        args = {**self.overrides, **custom, **kwargs, "mode": "export"}
        return ESP_Detect_Exporter(overrides=args, _callbacks=self.callbacks)(
            model=self.model
        )


model = ESP_YOLO("epoch130.pt")
for m in model.modules():

    if isinstance(m, Attention):
        #modifica il calcolo dell'attention, rendendolo ottimale per hardware esp-32.
        #Ad essere modificato è il forward del layer "Attention" del modulo C2PSA(model.10)
        #Cambia come l'attention calcola i pesi
        m.forward = ESP_Attention.forward.__get__(m)
    
    if isinstance(m, Detect):
        #Semplifica l'output del modello, spostando il post-processing al di fuori del modello
        #Ad essere modificato è l'output dell'intero modulo Detect (model.23)
        m.forward = ESP_Detect.forward.__get__(m)

#model.export(format="onnx", simplify=True, opset=13, dynamic=False, imgsz=96) #prima era imgsz=96  #DEFAULT!!
model.export(format="onnx", simplify=True, dynamic=False, opset=13, imgsz=64)

Ultralytics 8.3.174 🚀 Python-3.11.12 torch-2.8.0 CPU (Apple M1)
YOLO11micro summary (fused): 125 layers, 859,872 parameters, 0 gradients, 2.6 GFLOPs

PyTorch: starting from 'epoch130.pt' with input shape (1, 3, 64, 64) BCHW and output shape(s) ((1, 64, 8, 8), (1, 80, 8, 8), (1, 64, 4, 4), (1, 80, 4, 4), (1, 64, 2, 2), (1, 80, 2, 2)) (1.9 MB)

ONNX: starting export with onnx 1.17.0 opset 13...
ONNX: simplifying with onnxsim 0.4.36...
ONNX: export success ✅ 0.6s, saved as 'epoch130.onnx' (3.4 MB)

Export complete (0.7s)
Results saved to /Users/alessioprato/Desktop/Tesi Nuova/ESP32CAM_ESPIDF/Notebooks
Predict:         yolo predict task=detect model=epoch130.onnx imgsz=64  
Validate:        yolo val task=detect model=epoch130.onnx imgsz=64 data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


'epoch130.onnx'

In [ ]:
#QUANTIZZA MODELLO .ONNX INT8 IN FORMATO .ESPDL
import sys, os
import esp_ppq
sys.path.append(os.path.abspath("esp-detection"))
from deploy.quantize import quant_espdet
from esp_ppq.api import QuantizationSettingFactory

"""
# Configurazione specifica per ESP32-S3
quant_setting = QuantizationSettingFactory.espdl_setting()

# Forza scale uniforme per score e box
quant_setting.equalization = True
quant_setting.equalization_setting.opt_level = 1
"""

quant_espdet(
    onnx_path="epoch130.onnx",      # onnx appena esportato
    target="esp32s3",              
    num_of_bits=8,                 # quantizzazione int8
    device='cpu',
    batchsz=32,
    imgsz=[64, 64],              # deve combaciare con la dimensione del modello
    calib_dir="../../calibrazione/soloESP/",        #  cartella immagini reali
    espdl_model_path="yolo11n.espdl"
)


In [17]:
#ESPORTA MODELLO DA .PT IN .ONNX
from ultralytics import YOLO
model = YOLO("epoch130.pt")
model.export(format="onnx", imgsz=64) 

Ultralytics 8.3.174 🚀 Python-3.11.12 torch-2.8.0 CPU (Apple M1)
YOLO11micro summary (fused): 125 layers, 859,872 parameters, 0 gradients, 2.6 GFLOPs

PyTorch: starting from 'epoch130.pt' with input shape (1, 3, 64, 64) BCHW and output shape(s) (1, 84, 84) (1.9 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.63...
ONNX: export success ✅ 1.1s, saved as 'epoch130.onnx' (3.4 MB)

Export complete (1.4s)
Results saved to /Users/alessioprato/Desktop/Tesi Nuova/ESP32CAM_ESPIDF/Notebooks
Predict:         yolo predict task=detect model=epoch130.onnx imgsz=64  
Validate:        yolo val task=detect model=epoch130.onnx imgsz=64 data=/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


'epoch130.onnx'